In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD THE DATA
# We set low_memory=False because Stack Overflow data is notoriously messy
# with different data types in the same column.
print("Step 1: Loading raw dataset...")
df = pd.read_csv('results.csv', low_memory=False)

# 2. SELECT YOUR EXACT 22 COLUMNS
my_columns = [
    "Age", "Country", "EdLevel", "WorkExp", "YearsCode",
    "Employment", "RemoteWork", "OrgSize", "Industry", "ICorPM",
    "NewRole", "PurchaseInfluence", "LearnCodeAI", "ToolCountWork", "ToolCountPersonal",
    "AIThreat", "AIFrustration", "AISent", "AIComplex", "AIAgentChange",
    "ConvertedCompYearly", "JobSat"
]
df_master = df[my_columns].copy()
print(f"Step 2: Subset created. Initial shape: {df_master.shape}")

# 3. CLEAN THE TARGET METRIC FIRST
# If we don't know their JobSat, they are useless for a Burnout analysis.
df_master = df_master.dropna(subset=['JobSat'])
print(f"Step 3: JobSat nulls removed. New shape: {df_master.shape}")

# 4. FIX DATA TYPES (HANDLING TEXT HIDDEN IN NUMBER COLUMNS)
# Stack Overflow often has "Less than 1 year" or "More than 50 years" in number columns.
# We turn those into actual numbers so we can do math on them.
print("Step 4: Converting messy text-numbers to pure numbers...")
text_number_cols = ['WorkExp', 'YearsCode', 'ToolCountWork', 'ToolCountPersonal']

# Pre-cleaning the common "Less than..." and "More than..." text values
replacements = {
    'Less than 1 year': 0.5,
    'More than 50 years': 51,
    'Less than 1': 0.5, # Sometimes appears in ToolCount
    'More than 50': 51
}
for col in text_number_cols:
    df_master[col] = df_master[col].replace(replacements)

# Convert all text-number columns to floats, forcing errors to NaN so we can fill them later
for col in text_number_cols:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 5. HANDLE MISSING VALUES (IMPUTATION)
# Use the MEDIAN for numbers (Salary, Exp, Tools) because it is safe against outliers.
# Use 'Not Specified' for categories (Country, EdLevel, AIRelated, etc.) so we keep the row.
print("Step 5: Handling missing values via Imputation (Median and Placeholder)...")
numeric_cols = [
    'WorkExp', 'YearsCode', 'ToolCountWork', 'ToolCountPersonal',
    'ConvertedCompYearly', 'JobSat'
]
categorical_cols = df_master.select_dtypes(include=['object']).columns

# Handle Numbers with Median
for col in numeric_cols:
    # Ensure they are numbers first
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')
    median_val = df_master[col].median()
    df_master[col] = df_master[col].fillna(median_val)

# Handle Categories with 'Not Specified'
df_master[categorical_cols] = df_master[categorical_cols].fillna('Not Specified')

# 6. HANDLE OUTLIERS (SALARY CAP)
# Survey data is full of troll salaries (like $99 million).
# We must chop off the extreme top 2% so our charts are not warped.
print("Step 6: Capping extreme salary outliers at the 98th percentile...")
salary_cap = df_master['ConvertedCompYearly'].quantile(0.98)
df_master = df_master[df_master['ConvertedCompYearly'] <= salary_cap]

# 7. VERIFICATION & SAVING
print("\n--- FINAL DATASET VERIFICATION ---")
print(f"Final Gold Standard Rows: {df_master.shape[0]}")
print(f"Final Gold Standard Columns: {df_master.shape[1]} (All 22 present)")
print("Total Missing Values Left:", df_master.isnull().sum().sum())
print("-" * 50)

# Save the perfect, finalized dataset
df_master.to_csv('master_burnout_data.csv', index=False)
print("Success! Your complete dataset is saved as 'master_burnout_data.csv'")

Step 1: Loading raw dataset...
Step 2: Subset created. Initial shape: (49191, 22)
Step 3: JobSat nulls removed. New shape: (26670, 22)
Step 4: Converting messy text-numbers to pure numbers...
Step 5: Handling missing values via Imputation (Median and Placeholder)...
Step 6: Capping extreme salary outliers at the 98th percentile...

--- FINAL DATASET VERIFICATION ---
Final Gold Standard Rows: 26144
Final Gold Standard Columns: 22 (All 22 present)
Total Missing Values Left: 0
--------------------------------------------------
Success! Your complete dataset is saved as 'master_burnout_data.csv'


In [ ]:
dft = pd.read_csv('master_burnout_data.csv')
print(dft.head())

               Age      Country  \
0  25-34 years old      Ukraine   
1  25-34 years old  Netherlands   
2  35-44 years old      Ukraine   
3  35-44 years old      Ukraine   
4  35-44 years old      Ukraine   

                                           EdLevel  WorkExp  YearsCode  \
0  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)      8.0       14.0   
1              Associate degree (A.A., A.S., etc.)      2.0       10.0   
2     Bachelor’s degree (B.A., B.S., B.Eng., etc.)     10.0       12.0   
3     Bachelor’s degree (B.A., B.S., B.Eng., etc.)      4.0        5.0   
4  Master’s degree (M.A., M.S., M.Eng., MBA, etc.)     21.0       22.0   

                                          Employment  \
0                                           Employed   
1                                           Employed   
2  Independent contractor, freelancer, or self-em...   
3                                           Employed   
4  Independent contractor, freelancer, or self-em...   

       